# Extract virus detections (ENA / INSDC)

Downloads sequence **metadata** for the target vector-borne viruses from the [ENA Portal REST API](https://www.ebi.ac.uk/ena/portal/api/) and **overwrites** the local raw copy under `data/raw/virus/`.

- Source: INSDC via ENA (`result=sequence`, `tax_tree(<tax_id>)`)
- Endpoint: `https://www.ebi.ac.uk/ena/portal/api/search`
- License / terms: [EMBL-EBI Terms of Use](https://www.ebi.ac.uk/about/terms-of-use/) (no account required)
- Decision record: [`specs/02-virus-dataset-selection.md`](../specs/02-virus-dataset-selection.md)

This notebook stops at raw validation (TSV + manifest + smoke check). It does not clean, geocode, or join to VectorNet.

In [1]:
from __future__ import annotations

import hashlib
import io
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from tqdm import tqdm

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / "data" / "raw" / "virus"
TSV_NAME = "virus_detections.tsv"
TSV_PATH = RAW_DIR / TSV_NAME
MANIFEST_PATH = RAW_DIR / "manifest.json"

ENA_SEARCH_URL = "https://www.ebi.ac.uk/ena/portal/api/search"
CITATION = (
    "European Nucleotide Archive (ENA), EMBL-EBI. Sequence metadata retrieved via "
    "the ENA Portal API (INSDC: ENA + GenBank + DDBJ). "
    "https://www.ebi.ac.uk/ena/portal/api/"
)
DOWNLOAD_TIMEOUT_S = 600
CHUNK_SIZE = 1024 * 256
USER_AGENT = "vector-guard/0.1"

# Tax IDs verified against ENA Portal (tax_tree counts). Spec listed older/alternate
# IDs for CCHFV and Toscana that currently return empty sequence trees in ENA.
TARGETS: list[dict] = [
    {
        "pathogen": "West Nile virus",
        "tax_ids": [11082],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Usutu virus",
        "tax_ids": [64286],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Dengue virus",
        "tax_ids": [12637],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Chikungunya virus",
        "tax_ids": [37124],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Zika virus",
        "tax_ids": [64320],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Sindbis virus",
        "tax_ids": [11034],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Rift Valley fever virus",
        "tax_ids": [11588],
        "vector_group": "mosquito",
    },
    {
        "pathogen": "Tick-borne encephalitis virus",
        "tax_ids": [11084],
        "vector_group": "tick",
    },
    {
        "pathogen": "Crimean-Congo hemorrhagic fever virus",
        "tax_ids": [3052518],  # Orthonairovirus haemorrhagiae
        "vector_group": "tick",
    },
    {
        "pathogen": "Bluetongue virus",
        "tax_ids": [40051],
        "vector_group": "biting_midge",
    },
    {
        "pathogen": "Schmallenberg virus",
        "tax_ids": [1133363],
        "vector_group": "biting_midge",
    },
    {
        "pathogen": "Sandfly fever Naples phlebovirus / Toscana virus",
        "tax_ids": [11590, 206160, 1933189],  # Toscana + Naples complex
        "vector_group": "sandfly",
    },
]

ENA_FIELDS = [
    "accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "collection_date",
    "host",
    "isolation_source",
    "sample_accession",
    "study_accession",
]

RAW_DIR, ENA_SEARCH_URL, len(TARGETS)

(PosixPath('/workspaces/vector-guard/data/raw/virus'),
 'https://www.ebi.ac.uk/ena/portal/api/search',
 12)

## Wipe local raw directory

Every run replaces the previous local copy with the extract fetched now.

In [2]:
def wipe_raw_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    (path / ".gitkeep").touch(exist_ok=True)


wipe_raw_dir(RAW_DIR)
print(f"Wiped and recreated: {RAW_DIR}")

Wiped and recreated: /workspaces/vector-guard/data/raw/virus


## Fetch sequence metadata from ENA

For each target pathogen, query `tax_tree(<tax_id>)` with `limit=0` (all matching sequences). Keep rows that have at least a non-empty `country` or `collection_date`. Tag with `pathogen_name` and `vector_group`, then deduplicate by `accession`.

In [3]:
def ena_search_tsv(query: str, fields: list[str], timeout_s: int = DOWNLOAD_TIMEOUT_S) -> pd.DataFrame:
    """GET ENA Portal search results as a string-typed DataFrame."""
    params = {
        "result": "sequence",
        "query": query,
        "fields": ",".join(fields),
        "format": "tsv",
        "limit": "0",
    }
    url = f"{ENA_SEARCH_URL}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout_s) as response:
        chunks: list[bytes] = []
        while True:
            chunk = response.read(CHUNK_SIZE)
            if not chunk:
                break
            chunks.append(chunk)
    text = b"".join(chunks).decode("utf-8", errors="replace")
    if not text.strip():
        return pd.DataFrame(columns=fields)
    return pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)


def has_place_or_time(df: pd.DataFrame) -> pd.Series:
    country = df["country"].fillna("").str.strip()
    collected = df["collection_date"].fillna("").str.strip()
    return (country != "") | (collected != "")


frames: list[pd.DataFrame] = []
per_pathogen_counts: dict[str, dict[str, int]] = {}

for target in tqdm(TARGETS, desc="ENA pathogens"):
    pathogen = target["pathogen"]
    vector_group = target["vector_group"]
    tax_frames: list[pd.DataFrame] = []
    raw_rows = 0

    for tax_id in target["tax_ids"]:
        query = f"tax_tree({tax_id})"
        part = ena_search_tsv(query, ENA_FIELDS)
        if part.empty:
            continue
        part["query_tax_id"] = str(tax_id)
        tax_frames.append(part)
        raw_rows += len(part)

    if not tax_frames:
        per_pathogen_counts[pathogen] = {
            "raw_rows": 0,
            "kept_rows": 0,
            "tax_ids": target["tax_ids"],
        }
        continue

    combined = pd.concat(tax_frames, ignore_index=True)
    combined = combined.loc[has_place_or_time(combined)].copy()
    combined["pathogen_name"] = pathogen
    combined["vector_group"] = vector_group
    frames.append(combined)
    per_pathogen_counts[pathogen] = {
        "raw_rows": raw_rows,
        "kept_rows": len(combined),
        "tax_ids": target["tax_ids"],
    }

if frames:
    df = pd.concat(frames, ignore_index=True)
    before_dedupe = len(df)
    df = df.drop_duplicates(subset=["accession"], keep="first").reset_index(drop=True)
else:
    df = pd.DataFrame(
        columns=[*ENA_FIELDS, "query_tax_id", "pathogen_name", "vector_group"]
    )
    before_dedupe = 0

output_columns = [
    "accession",
    "pathogen_name",
    "vector_group",
    "query_tax_id",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "collection_date",
    "host",
    "isolation_source",
    "sample_accession",
    "study_accession",
]
df = df.reindex(columns=output_columns)

print(f"Rows before accession dedupe: {before_dedupe:,}")
print(f"Rows after accession dedupe:  {len(df):,}")
pd.DataFrame(per_pathogen_counts).T

ENA pathogens:   0%|          | 0/12 [00:00<?, ?it/s]

ENA pathogens:   8%|▊         | 1/12 [00:05<01:01,  5.60s/it]

ENA pathogens:  17%|█▋        | 2/12 [00:16<01:27,  8.70s/it]

ENA pathogens:  25%|██▌       | 3/12 [02:33<10:07, 67.47s/it]

ENA pathogens:  33%|███▎      | 4/12 [02:41<05:51, 43.98s/it]

ENA pathogens:  42%|████▏     | 5/12 [02:58<03:58, 34.05s/it]

... (6 outputs left out)

ENA pathogens: 100%|██████████| 12/12 [04:03<00:00, 10.42s/it]

ENA pathogens: 100%|██████████| 12/12 [04:03<00:00, 20.25s/it]

Rows before accession dedupe: 102,337
Rows after accession dedupe:  102,337


,raw_rows,kept_rows,tax_ids
West Nile virus,25302,8357,[11082]
Usutu virus,1934,1838,[64286]
Dengue virus,60223,57101,[12637]
Chikungunya virus,8122,6974,[37124]
Zika virus,3377,2244,[64320]
Sindbis virus,816,511,[11034]
Rift Valley fever virus,1849,1698,[11588]
Tick-borne encephalitis virus,3626,3440,[11084]
Crimean-Congo hemorrhagic fever virus,5370,5272,[3052518]
Bluetongue virus,14699,13952,[40051]


## Write TSV and manifest

In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


df.to_csv(TSV_PATH, sep="\t", index=False)

manifest = {
    "source_url": ENA_SEARCH_URL,
    "result": "sequence",
    "fetched_at": datetime.now(timezone.utc).isoformat(),
    "tsv_name": TSV_NAME,
    "tsv_bytes": TSV_PATH.stat().st_size,
    "sha256": sha256_file(TSV_PATH),
    "citation": CITATION,
    "fields": ENA_FIELDS,
    "filter": "keep rows with non-empty country or collection_date; dedupe by accession",
    "row_count": len(df),
    "per_pathogen": per_pathogen_counts,
    "targets": TARGETS,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Wrote {TSV_PATH} ({manifest['tsv_bytes']:,} bytes)")
print(f"Wrote {MANIFEST_PATH}")
manifest["sha256"], manifest["row_count"]

Wrote /workspaces/vector-guard/data/raw/virus/virus_detections.tsv (11,286,596 bytes)
Wrote /workspaces/vector-guard/data/raw/virus/manifest.json


('24acd29416f50f4993d6200b28c0a677b10a68f6d36cf00ecf17bc7463e3dd19', 102337)

## Smoke check

Reload the TSV from disk and confirm pathogen coverage and key columns.

In [5]:
print("Files in raw dir:")
for rel in sorted(
    p.relative_to(RAW_DIR).as_posix() for p in RAW_DIR.rglob("*") if p.is_file()
):
    print(f"  - {rel}")

loaded = pd.read_csv(TSV_PATH, sep="\t", dtype=str, keep_default_na=False)
print(f"\nCore file: {TSV_NAME}")
print(f"Rows: {len(loaded):,}")
print(f"Columns ({len(loaded.columns)}): {list(loaded.columns)}")
print("\nRows by pathogen_name:")
print(loaded["pathogen_name"].value_counts().to_string())
print("\nRows by vector_group:")
print(loaded["vector_group"].value_counts().to_string())
loaded.head(3)

Files in raw dir:
  - .gitkeep
  - manifest.json
  - virus_detections.tsv

Core file: virus_detections.tsv
Rows: 102,337
Columns (13): ['accession', 'pathogen_name', 'vector_group', 'query_tax_id', 'scientific_name', 'tax_id', 'country', 'location', 'collection_date', 'host', 'isolation_source', 'sample_accession', 'study_accession']

Rows by pathogen_name:
pathogen_name
Dengue virus                                        57101
Bluetongue virus                                    13952
West Nile virus                                      8357
Chikungunya virus                                    6974
Crimean-Congo hemorrhagic fever virus                5272
Tick-borne encephalitis virus                        3440
Zika virus                                           2244
Usutu virus                                          1838
Rift Valley fever virus                              1698
Sindbis virus                                         511
Sandfly fever Naples phlebovirus / Toscana vir

,accession,pathogen_name,vector_group,query_tax_id,scientific_name,tax_id,country,location,collection_date,host,isolation_source,sample_accession,study_accession
0,AF206518,West Nile virus,mosquito,11082,West Nile virus,11082,"USA:Connecticut, Greenwich-Stanford town line",,,Culex pipiens,,,
1,AY187014,West Nile virus,mosquito,11082,West Nile virus,11082,"USA:Harris County, TX",,,,,,
2,AY278442,West Nile virus,mosquito,11082,West Nile virus,11082,"Russia:Volgograd, low Volga",,,,human blood in 2000,,
